In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/cleaned-off/off_filtered.csv


In [2]:
!pip install -q faiss-cpu                                        # FAISS = fast similarity search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.5 MB/s eta 0:00:00


In [3]:
import time                                                      # timing
import requests                                                  # download from the internet
from io import BytesIO                                           # treat downloaded bytes like a file
from PIL import Image, ImageEnhance                              # open pictures + change brightness
from concurrent.futures import ThreadPoolExecutor                # many downloads at once

off_path = "/kaggle/input/cleaned-off/off_filtered.csv"                     # English OFF, copied from the Input panel
off = pd.read_csv(off_path, dtype={"code": str})                 # load; barcodes as text
off = off.reset_index(drop=True)                                 # rows numbered 0 to 71,297
print(off.shape)                                                 # expect (71298, 18)

(71298, 18)


In [4]:
import torch                                                     # PyTorch
from transformers import AutoImageProcessor, AutoModel           # Hugging Face loaders

device = "cuda" if torch.cuda.is_available() else "cpu"          # GPU if switched on
print(device)                                                    # must print "cuda"
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")  # image preparation for DINOv2
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device)     # the model, on the GPU
dino.eval()                                                      # use only, no training

cuda


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06,

In [5]:
headers = {"User-Agent": "FoodScanner student project (university coursework)"}  # tell OFF who is downloading

def fetch(url, tries=2):                                         # download one picture; None if it fails twice
    for attempt in range(tries):                                 # attempt 0, then attempt 1
        try:                                                     # try to download
            r = requests.get(url, headers=headers, timeout=10)   # fetch, wait at most 10 s
            r.raise_for_status()                                 # treat 404 etc. as an error
            img = Image.open(BytesIO(r.content)).convert("RGB")  # bytes -> picture
            if min(img.size) < 50:                               # tiny/broken picture (e.g. a 1-pixel placeholder)
                return None                                      # treat as failed
            return img                                           # normal picture: hand it back
        except Exception:                                        # failed this time
            time.sleep(1)                                        # wait 1 s before trying again (server hiccup)
    return None                                                  # failed both times

def alter(img):                                                  # same fixed changes as before
    w, h = img.size                                              # width, height
    img = img.crop((int(0.1 * w), int(0.1 * h), int(0.9 * w), int(0.9 * h)))  # cut 10% off every side
    img = img.rotate(10, expand=True, fillcolor=(255, 255, 255)) # tilt 10 degrees, white corners
    return ImageEnhance.Brightness(img).enhance(0.6)             # 40% darker

def embed(pictures):                                             # pictures -> rows of 384 numbers
    inputs = processor(images=pictures, return_tensors="pt").to(device)  # batch -> model input on GPU
    with torch.no_grad():                                        # no training
        vecs = dino(**inputs).pooler_output                      # batch -> (n, 384)
    vecs = torch.nn.functional.normalize(vecs, dim=1)            # each row length 1
    return vecs.cpu().numpy()                                    # back to numpy

In [6]:
def build(table, chunk=64, report_every=50):                     # download + embed a table, 64 at a time
    urls = table["image_url"].tolist()                           # links as a list
    orig, altered, ok_rows = [], [], []                          # originals, altered copies, rows that worked
    failed = 0                                                   # failure counter
    start = time.time()                                          # start clock
    with ThreadPoolExecutor(max_workers=16) as pool:             # 16 downloads at once
        for s in range(0, len(urls), chunk):                     # start of each chunk
            pics = list(pool.map(fetch, urls[s:s + chunk]))      # download this chunk in parallel
            good = [(s + j, p) for j, p in enumerate(pics) if p is not None]  # keep (row, picture) that worked
            failed += len(pics) - len(good)                      # count failures
            if good:                                             # at least one worked
                rows = [r for r, p in good]                      # their row numbers
                imgs = [p for r, p in good]                      # their pictures
                try:                                             # safety net for this batch
                    o = embed(imgs)                              # embed originals
                    a = embed([alter(p) for p in imgs])          # embed altered copies
                    orig.append(o)                               # store only if BOTH worked...
                    altered.append(a)                            # ...so originals and altered copies stay in step
                    ok_rows += rows                              # remember rows
                except Exception:                                # anything went wrong in this batch
                    failed += len(imgs)                          # count the whole batch as failed and carry on
            if (s // chunk) % report_every == 0:                 # every 50 chunks (3,200 pictures)...
                print(s + len(pics), "done,", failed, "failed,", round(time.time() - start), "s")  # ...progress
    print("finished in", round(time.time() - start), "s")        # total time
    emb = np.vstack(orig).astype("float32")                      # one table of originals
    emb_alt = np.vstack(altered).astype("float32")               # one table of altered copies
    return emb, emb_alt, np.array(ok_rows), failed               # hand back

emb_t, alt_t, rows_t, failed_t = build(off.head(128), report_every=1)  # small test on the first 128 products
print(emb_t.shape, alt_t.shape, "failed:", failed_t)             # expect about (128, 384) twice

64 done, 0 failed, 5 s
128 done, 0 failed, 10 s
finished in 10 s
(128, 384) (128, 384) failed: 0


In [7]:
emb, emb_alt, ok_rows, failed = build(off)                       # all 71,298 products
print(emb.shape, "failed:", failed)                              # how many worked / failed

np.save("off_image_embeddings.npy", emb)                         # the index vectors (the app needs this)
np.save("off_image_embeddings_altered.npy", emb_alt)             # altered copies (only for the test)
index_products = off.loc[ok_rows, ["code", "product_name", "image_url", "countries_en"]]  # which product each row is
index_products.to_csv("index_products.csv", index=False)         # row k of the embeddings = row k of this file
print("saved")                                                   # confirm

64 done, 0 failed, 5 s
3264 done, 2 failed, 235 s
6464 done, 6 failed, 470 s
9664 done, 8 failed, 700 s
12864 done, 8 failed, 928 s
16064 done, 9 failed, 1158 s
19264 done, 11 failed, 1438 s
22464 done, 12 failed, 1674 s
25664 done, 13 failed, 1900 s
28864 done, 14 failed, 2124 s
32064 done, 16 failed, 2351 s
35264 done, 20 failed, 2577 s
38464 done, 21 failed, 2804 s
41664 done, 22 failed, 3040 s
44864 done, 23 failed, 3281 s
48064 done, 25 failed, 3515 s
51264 done, 31 failed, 3758 s
54464 done, 33 failed, 3993 s
57664 done, 35 failed, 4234 s
60864 done, 37 failed, 4473 s
64064 done, 39 failed, 4708 s
67264 done, 55 failed, 4991 s
70464 done, 57 failed, 5228 s
finished in 5289 s
(71241, 384) failed: 57
saved


In [8]:
import faiss                                                     # similarity search

index = faiss.IndexFlatIP(384)                                   # exact search, cosine (length-1 vectors)
index.add(emb)                                                   # the whole catalogue
scores, found = index.search(emb_alt, 5)                         # each altered copy: 5 best matches
correct = np.arange(len(emb))                                    # altered row k should find row k
hit1 = found[:, 0] == correct                                    # best match right?
hit5 = (found == correct[:, None]).any(axis=1)                   # right one in top 5?
au_row = index_products["countries_en"].fillna("").str.contains("Australia").values  # Australian rows

results = pd.DataFrame({                                         # small results table
    "queries": ["all", "Australian"],                            # which group
    "n": [len(hit1), au_row.sum()],                              # how many queries
    "top1": [hit1.mean(), hit1[au_row].mean()],                  # top-1 accuracy
    "top5": [hit5.mean(), hit5[au_row].mean()],                  # top-5 accuracy
})
results.to_csv("retrieval_results.csv", index=False)             # save for the report
print(results.round(3))                                          # show it

      queries      n   top1   top5
0         all  71241  0.941  0.978
1  Australian   1991  0.942  0.980
